# FQL Succession Gate B — E-uni Sanity (Colab) [rev.3.1 notebook — real-time streaming + 2-seed + path-quote fix]

> 文档对应：
> - Spec：[`docs/fql_succession_p0p1_spec.md`](../docs/fql_succession_p0p1_spec.md) §5 (Task E — Gate B FQL E-uni sanity)
> - Plan：[`docs/fql_succession_plan_v0.md`](../docs/fql_succession_plan_v0.md) §3 (Lean MVP)
> - Dataset card：[`docs/fql_e_uni_anchor_dataset_card.md`](../docs/fql_e_uni_anchor_dataset_card.md)
> - **Interim report (rev.2 触发点)**：[`docs/fql_succession_gate_b_interim_report.md`](../docs/fql_succession_gate_b_interim_report.md) — rev.1 seed=42 单 seed 3/4 PASS marginal-fail，3 bugs 诊断完毕
> - Pre-flight Task A：[`docs/fql_audit_dryrun_report.md`](../docs/fql_audit_dryrun_report.md) (audit tool verified, Gate A.2 PASS on 200-ep dry-run)
> - Pre-flight Gate A.1：[`docs/rebrac_broad_validation_v2_report.md`](../docs/rebrac_broad_validation_v2_report.md) §2 (arrival_v2 reward bridge HOLDS at 0.85)
>
> **本 notebook rev.3.1 (2026-05-19)** — path-quote fix (Bug 5)：
> - **Bug 5 fix (rev.3.1 新增)**：§2 train / §3 eval 把 `--save-dir {ckpt_dir_str}` / `--checkpoint {ckpt_dir_str}` / `--output-json {test_json_str}` 三处都用单引号包起来。Colab 项目路径 `/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5/` 含**空格**（"Colab Notebooks"），未加引号时 Jupyter `{var}` 内插后 shell 按空格分词，argparse 拒收尾部，rev.3 实测 `[FAIL] rebrac seed=0 agent_final.pt missing after 0.4 min` (`unrecognized arguments: Notebooks/...`)。seed=42 两 run 没炸只是因为它们走 Python `Path.exists()` skip-resume 没过 shell
>
> **rev.3 (2026-05-19) 沿用 (未变)**：
> - **Bug 4 fix**：§2 train / §3 eval 从 `os.system(cmd_str)` 改为 Jupyter `!python ... \` magic + `{var}` 内插。`os.system` 在 Colab 实测**不 stream**（rev.1+rev.2 FQL 训练 49 min 全程空白），切换到 `!python` magic 后训练 loss/eval 逐行实时打到 cell（早期 `rebrac_c1_*` notebook 一直用此 pattern）
> - **代价**：~30 行 common flag 在 rebrac/fql 两 if/elif 分支重复，无 `_common_cli` 抽象 —— 换 stream 的可见性，trade-off 合理
>
> **rev.2 (2026-05-19) 沿用 (未变)**：
> - **Bug 1 fix**：§5 verdict `_dedup_by_train_step()` 处理 append-mode log 污染
> - **Bug 3 fix**：§5.5 字段名 `eval_avg_*` → `eval_*`
> - **Option B 扩展**：`SEEDS = [42, 0]` 4 runs = 2 algo × 2 seed
>
> **Bug 2 (留 next，未在本 rev 处理)**：`--episodes 100` 被 manifest 静默 override 至 30 ep；verdict 报告 noise floor 给上下文

目的：在 E-uni 1000-ep paper anchor 上跑 **paired** ReBRAC + FQL × 2 seed sanity，应用 spec §5.3 四条 Gate B 判据（aggregated across seeds）决定是否进入 P2 main comparison。

## 执行摘要（4 paired runs，2 seed × 2 algo）

| Run | Algo | Dataset | Manifest | Steps | Seed | Status | Notes |
|---|---|---|---|---|---|---|---|
| 1 | **ReBRAC** | `privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000` | `single_u10_cross_tgt15` | 200k | 42 | rev.1 done | skip-resume |
| 2 | **FQL** | 同上 | 同上 | 200k | 42 | rev.1 done (logs 污染) | skip-resume + §5 dedup |
| 3 | **ReBRAC** | 同上 | 同上 | 200k | 0 | rev.3.1 new | fresh run，real-time stream + path-quote |
| 4 | **FQL** | 同上 | 同上 | 200k | 0 | rev.3.1 new | fresh run，real-time stream + path-quote |

**Wallclock 预算**：seed=42 两 run 已完成 (~75 min)，本 rev 只需跑 seed=0 两 run ≈ **75 min L4**（ReBRAC ~25 + FQL ~50）。

## 输出位置

| 类型 | 路径 | Sync 回 git? |
|---|---|---|
| Train artifacts | `checkpoints/offline/fql_succession/gate_b/<algo>_e_uni_seed<n>/{agent_final.pt, trainer_state.json, train_log.jsonl, eval_log.csv}` | 否（大文件，gitignore） |
| Final eval | `results/offline/fql_succession/gate_b/<algo>_e_uni_seed<n>/test_result.json` | **是** |
| In-training eval traj | `results/offline/fql_succession/gate_b/<algo>_e_uni_seed<n>/eval_log.csv`（从 checkpoint dir copy） | **是** |
| 汇总 | `results/offline/fql_succession/gate_b/summaries/{gate_b_verdict.json, gate_b_overview.csv}` | **是** |

## Gate B 判据 (spec §5.3, pre-registered; rev.2 across 2 seeds per algo aggregated mean)

| # | 指标 | 通过条件 |
|---|---|---|
| c1 | FQL `test_success` last 3 eval mean（across-seed mean） | **≥ ReBRAC same − 0.08** |
| c2 | FQL `loss_flow` 末段（across-seed mean of last 5% train log） | **< 0.05**（teacher 收敛） |
| c3 | FQL `actor_loss` 末段数量级 | **vs ReBRAC same，比值在 [0.1, 10]** |
| c4 | FQL eval 曲线 monotonicity（across-seed mean of last 30% slope） | **slope ≥ 0**（no collapse） |

**4 条全过 → Gate B pass → 进入 P2 main comparison spec 写作。** 任一失败按 spec §5.3 / interim report §8 mitigation 处理。

## 已知 noise / contamination caveat

| Source | 描述 | 处理 |
|---|---|---|
| Bug 1 | rev.1 FQL seed=42 因 train script append-mode + skip-resume race 导致 `train_log.jsonl` (386 行) 和 `eval_log.csv` (38 行) 跨 run 拼接 | rev.2 §5 verdict `_dedup_by_train_step()` 去重 |
| Bug 2 | `--manifest` override `--episodes` → 实际 30 ep eval（CI ±8.4pp/eval） | verdict 报告中 noted；P2 spec 写作时考虑大 manifest 或调整 c1/c4 阈值 |
| Bug 4 | rev.1 / rev.2 用 `os.system` Colab 实测**不 stream**（cell 训练时全程空白） | rev.3 改用 Jupyter `!python` magic，逐行实时输出 |
| Bug 5 | rev.3 `--save-dir {ckpt_dir_str}` 内插后 Colab 路径含空格被 shell 分词 → argparse fail | rev.3.1 给 3 处路径 `{var}` 包单引号 |
| Single-seed power | rev.1 仅 seed=42，c4 单点 noise 不可分辨 | rev.2 双 seed = 标准方差估计起点 |

## Spec CLI ↔ 实际代码 字段映射（rev.1 不动 spec，rev.2/rev.3 沿用）

本 notebook 用**实际代码**字段名；spec 写作时的字段名已 deprecated 但 spec 待 Task E 闭环统一更新到 v1.1。

| Spec 名称 | 实际名称 |
|---|---|
| `--algorithm` | `--algo` |
| `--eval-every-steps` | `--eval-every` |
| `--rebrac-actor-bc-coef` | `--actor-penalty-coef` |
| `--rebrac-critic-bc-coef` | `--critic-penalty-coef` |
| `--rebrac-critic-use-layernorm` | `--critic-layernorm` |
| `--fql-flow-steps` | `--flow-steps` |
| `--fql-distill-alpha-bc` | `--distill-alpha-bc` |

Dataset path：spec 的 placeholder `offline_data/fql_succession/e_uni_1000/` → 实际采用现有 naming convention `offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/`。

## 0. 环境 sanity

In [ ]:
!lscpu | head -8
print()
!nvidia-smi

In [ ]:
import torch
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device 0      : {torch.cuda.get_device_name(0)}')
    print(f'cuDNN         : {torch.backends.cudnn.version()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

In [ ]:
!pwd
!ls scripts/train_offline.py scripts/evaluate_offline.py
!ls docs/fql_succession_p0p1_spec.md docs/fql_e_uni_anchor_dataset_card.md
!ls auv_nav/fql.py auv_nav/rebrac.py
!ls offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz \
    offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/metadata.json
!ls benchmarks/single_u10_cross_tgt15.json
!ls wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi_meta.json

In [ ]:
# Optional sanity: confirm FQL unit tests still pass on the Colab runtime
# (skip with shift+enter if you've already verified locally).
!python -m pytest tests/test_fql.py -q --no-header 2>&1 | tail -20

## 1. Run matrix

4 runs = ReBRAC + FQL × seeds=[42, 0]。所有 run 共用：
- 同 dataset（E-uni 1000-ep privileged）
- 同 eval manifest（`single_u10_cross_tgt15`，实际 30 ep — Bug 2 caveat）
- 同 total-steps（200k）
- 同 eval-every（10k step → 20 个 eval point per run）
- 同 sampling-mode（`uniform`，spec §5.2；与 broad val v2 `shuffle_no_replacement` 不同）

差异只在 algo + algo-specific CLI flags：
- ReBRAC：`--actor-penalty-coef 4.0 --critic-penalty-coef 2.0 --critic-layernorm --no-actor-layernorm`
- FQL：`--flow-steps 10 --distill-alpha-bc 1.0`（teacher_lr / time_embed_dim 用默认 3e-4 / 32）

Skip-resume：
- **train 阶段** 看 `<ckpt_dir>/trainer_state.json` + `<ckpt_dir>/agent_final.pt` 都存在则跳过
- **eval 阶段** 看 `<result_dir>/test_result.json` 存在则跳过

rev.1 已完成的 (ReBRAC, seed=42) + (FQL, seed=42) 走 skip-resume；本 rev 新跑 (ReBRAC, seed=0) + (FQL, seed=0)。

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path('.').resolve()
CKPT_ROOT = REPO_ROOT / 'checkpoints' / 'offline' / 'fql_succession' / 'gate_b'
RESULT_ROOT = REPO_ROOT / 'results' / 'offline' / 'fql_succession' / 'gate_b'
SUMMARIES_DIR = RESULT_ROOT / 'summaries'
SUMMARIES_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = 'offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000'
DATASET_NPZ = f'{DATASET_DIR}/transitions.npz'
FLOW_PATH = 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
MANIFEST = 'benchmarks/single_u10_cross_tgt15.json'
SEEDS = [42, 0]  # rev.2: seed=42 carried over from rev.1, seed=0 added for Option B power
TOTAL_STEPS = 200_000
EVAL_EVERY = 10_000
EVAL_EPISODES = 100  # CLI value; effective is min(this, manifest_size) — see Bug 2 caveat

ALGOS = ['rebrac', 'fql']

RUNS = []
for seed in SEEDS:
    for algo in ALGOS:
        run_id = f'{algo}_e_uni_seed{seed}'
        ckpt_dir = CKPT_ROOT / run_id
        result_dir = RESULT_ROOT / run_id
        RUNS.append({
            'run_id': run_id,
            'algo': algo,
            'seed': seed,
            'dataset': DATASET_NPZ,
            'manifest': MANIFEST,
            'flow': FLOW_PATH,
            'ckpt_dir': str(ckpt_dir),
            'result_dir': str(result_dir),
        })

print(f'{"#":>2} {"algo":<8} {"seed":>5}  ckpt_dir / result_dir')
print('-' * 110)
for i, r in enumerate(RUNS, 1):
    print(f'{i:>2} {r["algo"]:<8} {r["seed"]:>5}  ckpt:   {r["ckpt_dir"]}')
    print(f'   {"":<8} {"":>5}  result: {r["result_dir"]}')


## 2. Training (4 runs, ~75 min L4 — 2 已完成 skip + 2 fresh)

**rev.3 关键改动 (Bug 4 fix)**：本 cell 从 `os.system(cmd_str)` 改为 Jupyter `!python ... \` magic + `{var}` 内插，让训练 loss / in-training eval 逐行实时打到 cell（早期 `rebrac_c1_*` notebook 一直用此 pattern；`os.system` 在 Colab 实测**不 stream**）。代价：~30 行 common flag 在 rebrac/fql 两 if/elif 分支重复。

训练时**保留** `--eval-every 10000 --eval-episodes 100`，Gate B c1/c4 需要 in-training eval 轨迹。

训练输出文件：
- `<ckpt_dir>/train_log.jsonl`：每 `--log-every`（默认 1000 step）的 loss metrics（含 FQL `loss_flow`）
- `<ckpt_dir>/eval_log.csv`：每 `--eval-every` 的 in-training eval（实际 30 ep — Bug 2）→ Gate B c1/c4 数据源
- `<ckpt_dir>/agent_final.pt`：最终 checkpoint
- `<ckpt_dir>/trainer_state.json`：完整 trainer state（含 algo / agent_config，用于 evaluate_offline 自动 dispatch）

Skip-resume：先看 `<ckpt_dir>/trainer_state.json + agent_final.pt` 都在则跳过 — rev.1 已完成的 (ReBRAC, seed=42) + (FQL, seed=42) 走此路径；本 rev 实跑 seed=0 两 run。

In [ ]:
import time
from pathlib import Path

for i, r in enumerate(RUNS, 1):
    algo = r['algo']
    seed = r['seed']
    ckpt_dir = Path(r['ckpt_dir'])
    ckpt_dir_str = str(ckpt_dir)
    trainer_state = ckpt_dir / 'trainer_state.json'
    agent_final = ckpt_dir / 'agent_final.pt'

    # Per-run vars (used by !python {var} interpolation below)
    DATASET = r['dataset']
    FLOW = r['flow']
    MFST = r['manifest']

    print(f'\n========== [{i}/{len(RUNS)}] train {algo} seed={seed} ==========')

    if trainer_state.exists() and agent_final.exists():
        print(f'[skip] already complete: {ckpt_dir}')
        continue

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    # rev.3: Jupyter !python magic for real-time streaming (was os.system in rev.2).
    # rev.3.1 (Bug 5 fix): single-quote {ckpt_dir_str} because Colab path
    #   `/content/drive/MyDrive/Colab Notebooks/...` has a SPACE — without quotes
    #   the shell splits the arg and argparse rejects the tail as positional.
    # Per-algo branches duplicate common flags — trade-off for early-style streaming.
    if algo == 'rebrac':
        !python -m scripts.train_offline \
            --algo rebrac \
            --offline-data {DATASET} \
            --flow {FLOW} \
            --manifest {MFST} \
            --probe-layout s0 \
            --history-length 4 \
            --task-geometry cross_stream \
            --target-speed 1.5 \
            --objective arrival_v2 \
            --total-steps {TOTAL_STEPS} \
            --batch-size 256 \
            --sampling-mode uniform \
            --hidden-dim 256 \
            --num-hidden-layers 3 \
            --actor-lr 3e-4 \
            --critic-lr 3e-4 \
            --gamma 0.99 \
            --tau 0.005 \
            --policy-noise 0.2 \
            --noise-clip 0.5 \
            --policy-freq 2 \
            --grad-clip-norm 10.0 \
            --normalizer-eps 1e-3 \
            --eval-every {EVAL_EVERY} \
            --eval-episodes {EVAL_EPISODES} \
            --eval-workers 4 \
            --eval-worker-device cpu \
            --log-every 1000 \
            --checkpoint-every 0 \
            --seed {seed} \
            --device cuda \
            --save-dir '{ckpt_dir_str}' \
            --actor-penalty-coef 4.0 \
            --critic-penalty-coef 2.0 \
            --critic-layernorm \
            --no-actor-layernorm
    elif algo == 'fql':
        !python -m scripts.train_offline \
            --algo fql \
            --offline-data {DATASET} \
            --flow {FLOW} \
            --manifest {MFST} \
            --probe-layout s0 \
            --history-length 4 \
            --task-geometry cross_stream \
            --target-speed 1.5 \
            --objective arrival_v2 \
            --total-steps {TOTAL_STEPS} \
            --batch-size 256 \
            --sampling-mode uniform \
            --hidden-dim 256 \
            --num-hidden-layers 3 \
            --actor-lr 3e-4 \
            --critic-lr 3e-4 \
            --gamma 0.99 \
            --tau 0.005 \
            --policy-noise 0.2 \
            --noise-clip 0.5 \
            --policy-freq 2 \
            --grad-clip-norm 10.0 \
            --normalizer-eps 1e-3 \
            --eval-every {EVAL_EVERY} \
            --eval-episodes {EVAL_EPISODES} \
            --eval-workers 4 \
            --eval-worker-device cpu \
            --log-every 1000 \
            --checkpoint-every 0 \
            --seed {seed} \
            --device cuda \
            --save-dir '{ckpt_dir_str}' \
            --flow-steps 10 \
            --distill-alpha-bc 1.0 \
            --teacher-lr 3e-4 \
            --flow-time-embed-dim 32
    else:
        raise ValueError(f'Unknown algo: {algo}')

    elapsed = (time.time() - t0) / 60
    if agent_final.exists():
        print(f'[done] {algo} seed={seed} → {ckpt_dir} ({elapsed:.1f} min)')
    else:
        print(f'\n[FAIL] {algo} seed={seed} agent_final.pt missing after {elapsed:.1f} min')
        break


## 3. Final evaluation (2 runs × 100 episodes against fixed manifest)

训练内 eval 已经写在 `eval_log.csv`（最后一行 = train_step=200000 的 eval），这里 separately 再跑一次 `scripts.evaluate_offline` 作为 **canonical final eval**（与 broad val v2 protocol 一致）。

`evaluate_offline` 通过 `trainer_state.json` 自动 dispatch 到正确的 agent class（rebrac / fql），无需手动指定 algo。

In [ ]:
from pathlib import Path

for i, r in enumerate(RUNS, 1):
    algo = r['algo']
    seed = r['seed']
    ckpt_dir = Path(r['ckpt_dir'])
    result_dir = Path(r['result_dir'])
    test_json = result_dir / 'test_result.json'
    ckpt_dir_str = str(ckpt_dir)
    test_json_str = str(test_json)
    MFST = r['manifest']

    print(f'\n========== [{i}/{len(RUNS)}] eval {algo} seed={seed} ==========')

    if test_json.exists():
        print(f'[skip] test_result.json exists: {test_json}')
        continue
    if not (ckpt_dir / 'agent_final.pt').exists():
        print(f'[skip] no agent_final.pt at {ckpt_dir} (training not done?)')
        continue

    result_dir.mkdir(parents=True, exist_ok=True)

    # rev.3: Jupyter !python magic for real-time streaming (was os.system in rev.2).
    # rev.3.1 (Bug 5 fix): single-quote {ckpt_dir_str} / {test_json_str} because
    #   Colab path `/content/drive/MyDrive/Colab Notebooks/...` has a SPACE — without
    #   quotes the shell splits the arg and argparse rejects the tail as positional.
    # evaluate_offline auto-dispatches via trainer_state.json (no per-algo branching).
    !python -m scripts.evaluate_offline \
        --checkpoint '{ckpt_dir_str}' \
        --agent-file agent_final.pt \
        --manifest {MFST} \
        --episodes 100 \
        --seed 123 \
        --device cuda \
        --num-workers 4 \
        --worker-device cpu \
        --output-json '{test_json_str}'

    if test_json.exists():
        print(f'[done] {test_json}')
    else:
        print(f'[FAIL] {algo} seed={seed} test_result.json missing after eval')
        break


## 3.5 Archive in-training eval_log.csv → results/

Gate B c1 (last 3 eval mean) + c4 (last 30% monotonicity) 都依赖 `eval_log.csv`。`eval_log.csv` 由训练写入 checkpoint dir，但 checkpoint dir 是 gitignored —— 把它 copy 到 `results/` 一份，sync 回 git 留痕。

In [ ]:
import shutil
for r in RUNS:
    src = Path(r['ckpt_dir']) / 'eval_log.csv'
    dst = Path(r['result_dir']) / 'eval_log.csv'
    if not src.exists():
        print(f'[noop] {src} missing (training not done)')
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'[archive] {src} → {dst}')

## 4. Raw diagnostic — test_result.json 字段 + eval_log.csv 末 3 行 + train_log.jsonl 末 5 行

在跑 §5 verdict 前打印 raw 数字，便于排查 verdict 出 bug 时定位是哪一步出问题。

In [ ]:
import csv
import json

for r in RUNS:
    print(f'\n=== {r["algo"]} seed={r["seed"]} ===')

    # 4a. test_result.json (final 100-ep eval)
    test_json = Path(r['result_dir']) / 'test_result.json'
    print(f'\n[a] test_result.json @ {test_json}')
    if not test_json.exists():
        print('  MISSING')
    else:
        d = json.loads(test_json.read_text())
        for k in sorted(d.keys()):
            v = d[k]
            if isinstance(v, (int, float, str, bool, type(None))):
                print(f'  {k}: {v}')
            elif isinstance(v, list):
                print(f'  {k}: list(len={len(v)})')
            elif isinstance(v, dict):
                print(f'  {k}: dict(keys={list(v.keys())[:5]}{"..." if len(v) > 5 else ""})')

    # 4b. eval_log.csv 末 3 行
    eval_csv = Path(r['result_dir']) / 'eval_log.csv'
    if not eval_csv.exists():
        eval_csv = Path(r['ckpt_dir']) / 'eval_log.csv'
    print(f'\n[b] eval_log.csv (last 3) @ {eval_csv}')
    if not eval_csv.exists():
        print('  MISSING')
    else:
        with eval_csv.open() as fp:
            rows = list(csv.DictReader(fp))
        for row in rows[-3:]:
            step = row.get('train_step', '?')
            succ = row.get('eval_success_rate', '?')
            ret_ = row.get('eval_return', '?')
            prog = row.get('eval_progress_ratio', '?')
            print(f'  step={step}  success={succ}  return={ret_}  progress={prog}')

    # 4c. train_log.jsonl 末 5 行（FQL 看 loss_flow，ReBRAC 看 loss_actor）
    train_jsonl = Path(r['ckpt_dir']) / 'train_log.jsonl'
    print(f'\n[c] train_log.jsonl (last 5) @ {train_jsonl}')
    if not train_jsonl.exists():
        print('  MISSING')
    else:
        with train_jsonl.open() as fp:
            lines = fp.readlines()
        for line in lines[-5:]:
            row = json.loads(line)
            step = row.get('train_step', '?')
            keys_of_interest = ['critic_loss', 'actor_loss', 'bc_loss', 'mean_q', 'lambda',
                                 'loss_flow', 'td_abs_error', 'critic_penalty']
            vals = ' '.join(
                f'{k}={row[k]:.4f}' if isinstance(row.get(k), (int, float)) else ''
                for k in keys_of_interest if k in row
            )
            print(f'  step={step}  {vals.strip()}')

## 5. Gate B verdict — 4 criteria (rev.2: 2-seed aggregated + log dedup)

判据来源 spec §5.3（pre-registered）；rev.2 扩展为多 seed aggregation。

**rev.2 关键改动 (Bug 1 fix)**：本 cell 加 `_dedup_by_train_step()` helper，对 `eval_log.csv` / `train_log.jsonl` 按 `train_step` 去重，**保留最后一次出现**（last write wins）。这修复 rev.1 因 train script append-mode + skip-resume race 导致的跨 run 拼接污染（rev.1 FQL seed=42 eval log 38 行实际是同一 seed 两次 run 拼接）。

| # | 指标 | 数据源 | 通过条件 (across-seed mean) |
|---|---|---|---|
| c1 | FQL last-3-eval success mean | `eval_log.csv` 末 3 行 dedup 后 `eval_success_rate` mean | ≥ ReBRAC same − 0.08 |
| c2 | FQL loss_flow 末段 | `train_log.jsonl` 末 5% 行 dedup 后 `loss_flow` mean | < 0.05 |
| c3 | FQL actor_loss 末段 OOM | `train_log.jsonl` 末 5% 行 dedup 后 `actor_loss` mean | vs ReBRAC same，比值在 [0.1, 10] |
| c4 | FQL eval 曲线 monotonicity | `eval_log.csv` 末 30% 行 dedup 后 `eval_success_rate` 线性 slope | slope ≥ 0 |

**Aggregation**：每个 seed 单独算 4 个指标 → 同 algo 跨 seed mean → 应用阈值。

**4 条全过 → Gate B PASS → 进入 P2 main comparison spec**。任一失败 → 按 spec §5.3 + interim report §8 mitigation 处理。

In [ ]:
import csv
import json
import math
import statistics
from collections import defaultdict


def _dedup_by_train_step(rows, key='train_step'):
    """Bug 1 fix: keep the last occurrence per train_step (handles append-mode contamination
    from re-runs where the training script appended to existing logs without truncation).
    Returns rows sorted by train_step ascending.
    """
    seen = {}
    for row in rows:
        try:
            step = int(float(row.get(key, -1)))
        except (TypeError, ValueError):
            continue
        seen[step] = row  # last write wins
    return [seen[s] for s in sorted(seen.keys())]


def _load_eval_log(run):
    """Return dedup-by-train_step list of dicts from eval_log.csv."""
    for candidate in [Path(run['result_dir']) / 'eval_log.csv',
                       Path(run['ckpt_dir']) / 'eval_log.csv']:
        if candidate.exists():
            with candidate.open() as fp:
                rows = list(csv.DictReader(fp))
            return _dedup_by_train_step(rows)
    return []


def _load_train_jsonl(run):
    path = Path(run['ckpt_dir']) / 'train_log.jsonl'
    if not path.exists():
        return []
    with path.open() as fp:
        rows = [json.loads(line) for line in fp if line.strip()]
    return _dedup_by_train_step(rows)


def _safe_float(x):
    try:
        f = float(x)
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None


def _linear_slope(ys):
    n = len(ys)
    if n < 2:
        return 0.0
    xs = list(range(n))
    mean_x = sum(xs) / n
    mean_y = sum(ys) / n
    num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    den = sum((x - mean_x) ** 2 for x in xs)
    return num / den if den > 0 else 0.0


def _per_seed_metrics(run):
    eval_rows = _load_eval_log(run)
    train_rows = _load_train_jsonl(run)

    succ_traj = [_safe_float(row.get('eval_success_rate')) for row in eval_rows]
    succ_traj = [s for s in succ_traj if s is not None]
    last3_mean = statistics.mean(succ_traj[-3:]) if len(succ_traj) >= 3 else (
        statistics.mean(succ_traj) if succ_traj else float('nan'))
    last30pct_idx = max(1, int(0.7 * len(succ_traj)))
    last30pct_slope = _linear_slope(succ_traj[last30pct_idx:]) if succ_traj else 0.0

    # 末段 = 末 5% rows of (dedup'd) train_log.jsonl
    tail_n = max(1, int(0.05 * len(train_rows)))
    tail = train_rows[-tail_n:]
    loss_flow_tail = [_safe_float(row.get('loss_flow')) for row in tail]
    loss_flow_tail = [v for v in loss_flow_tail if v is not None]
    actor_loss_tail = [_safe_float(row.get('actor_loss')) for row in tail]
    actor_loss_tail = [v for v in actor_loss_tail if v is not None]

    return {
        'seed': run['seed'],
        'n_eval_points_dedup': len(succ_traj),
        'eval_success_trajectory': succ_traj,
        'last3_eval_success_mean': last3_mean,
        'last30pct_eval_slope': last30pct_slope,
        'n_train_log_rows_dedup': len(train_rows),
        'loss_flow_last_5pct_mean': statistics.mean(loss_flow_tail) if loss_flow_tail else float('nan'),
        'actor_loss_last_5pct_mean': statistics.mean(actor_loss_tail) if actor_loss_tail else float('nan'),
    }


# ---- Extract per-seed metrics, then aggregate by algo ----
per_algo = defaultdict(dict)
for r in RUNS:
    per_algo[r['algo']][r['seed']] = _per_seed_metrics(r)


def _agg(values):
    """Across-seed mean over non-nan values; nan if all missing."""
    clean = [v for v in values if not (isinstance(v, float) and math.isnan(v))]
    return statistics.mean(clean) if clean else float('nan')


agg_by_algo = {}
for algo, by_seed in per_algo.items():
    seeds = sorted(by_seed.keys())
    agg_by_algo[algo] = {
        'seeds': seeds,
        'n_seeds': len(seeds),
        'last3_mean': _agg([by_seed[s]['last3_eval_success_mean'] for s in seeds]),
        'last30pct_slope_mean': _agg([by_seed[s]['last30pct_eval_slope'] for s in seeds]),
        'loss_flow_mean': _agg([by_seed[s]['loss_flow_last_5pct_mean'] for s in seeds]),
        'actor_loss_mean': _agg([by_seed[s]['actor_loss_last_5pct_mean'] for s in seeds]),
    }


# ---- Pretty-print per-seed + per-algo ----
print('=' * 110)
print('Gate B raw inputs (per-seed, dedup applied):')
for algo in ['rebrac', 'fql']:
    print(f'\n  {algo}:')
    for seed, s in sorted(per_algo[algo].items()):
        print(f'    seed={seed}:')
        print(f'      n_eval_points_dedup    = {s["n_eval_points_dedup"]}')
        print(f'      last3_eval_success     = {s["last3_eval_success_mean"]:.4f}')
        print(f'      last30pct_eval_slope   = {s["last30pct_eval_slope"]:+.5f}')
        print(f'      n_train_log_rows_dedup = {s["n_train_log_rows_dedup"]}')
        print(f'      loss_flow_last_5pct    = {s["loss_flow_last_5pct_mean"]:.4f}')
        print(f'      actor_loss_last_5pct   = {s["actor_loss_last_5pct_mean"]:+.4f}')

print('\n' + '=' * 110)
print('Gate B aggregated (per-algo across-seed mean):')
for algo in ['rebrac', 'fql']:
    a = agg_by_algo[algo]
    print(f'\n  {algo} ({a["n_seeds"]} seeds {a["seeds"]}):')
    print(f'    last3_mean           = {a["last3_mean"]:.4f}')
    print(f'    last30pct_slope_mean = {a["last30pct_slope_mean"]:+.5f}')
    print(f'    loss_flow_mean       = {a["loss_flow_mean"]:.4f}')
    print(f'    actor_loss_mean      = {a["actor_loss_mean"]:+.4f}')


# ---- Apply Gate B criteria on aggregated values ----
rebrac = agg_by_algo.get('rebrac', {})
fql = agg_by_algo.get('fql', {})

rebrac_last3 = rebrac.get('last3_mean', float('nan'))
fql_last3 = fql.get('last3_mean', float('nan'))
fql_loss_flow = fql.get('loss_flow_mean', float('nan'))
fql_actor_loss = fql.get('actor_loss_mean', float('nan'))
rebrac_actor_loss = rebrac.get('actor_loss_mean', float('nan'))
fql_slope = fql.get('last30pct_slope_mean', 0.0)

# c1: FQL last-3 ≥ ReBRAC last-3 − 0.08
c1_pass = (fql_last3 >= rebrac_last3 - 0.08) if not (math.isnan(fql_last3) or math.isnan(rebrac_last3)) else False
c1_gap = fql_last3 - rebrac_last3

# c2: FQL loss_flow < 0.05
c2_pass = (fql_loss_flow < 0.05) if not math.isnan(fql_loss_flow) else False

# c3: FQL/ReBRAC actor_loss within ±1 OOM
if not math.isnan(fql_actor_loss) and not math.isnan(rebrac_actor_loss) and abs(rebrac_actor_loss) > 1e-9:
    ratio = abs(fql_actor_loss) / abs(rebrac_actor_loss)
    c3_pass = (0.1 <= ratio <= 10.0)
else:
    ratio = float('nan')
    c3_pass = False

# c4: FQL last 30% eval slope ≥ 0
c4_pass = fql_slope >= 0

verdict_block = {
    'c1_last3_parity': {
        'fql_last3_mean': fql_last3, 'rebrac_last3_mean': rebrac_last3,
        'gap_fql_minus_rebrac': c1_gap,
        'threshold': -0.08, 'pass': bool(c1_pass),
    },
    'c2_loss_flow_converged': {
        'fql_loss_flow_mean': fql_loss_flow,
        'threshold': 0.05, 'pass': bool(c2_pass),
    },
    'c3_actor_loss_oom_parity': {
        'fql_actor_loss_mean': fql_actor_loss,
        'rebrac_actor_loss_mean': rebrac_actor_loss,
        'abs_ratio_fql_over_rebrac': ratio,
        'threshold': '[0.1, 10.0]', 'pass': bool(c3_pass),
    },
    'c4_eval_monotonicity_last_30pct': {
        'fql_slope_mean': fql_slope, 'threshold': 0.0, 'pass': bool(c4_pass),
    },
    'overall_pass': bool(c1_pass and c2_pass and c3_pass and c4_pass),
}

summary = {
    'per_seed': {algo: per_algo[algo] for algo in per_algo},
    'aggregated': agg_by_algo,
    'verdict': verdict_block,
    'notes': {
        'bug1_dedup_applied': True,
        'bug2_caveat': 'eval used manifest size (30 ep) instead of CLI --episodes 100; CI ±8.4pp/eval',
        'aggregation': 'per-algo mean across seeds; verdict on aggregated mean (looser than per-seed AND)',
    },
}

print('\n' + '=' * 110)
print('Gate B verdict:')
print(f'  c1 last-3 parity:        FQL={fql_last3:.4f} vs ReBRAC={rebrac_last3:.4f} '
      f'(Δ={c1_gap:+.4f}, threshold ≥ −0.08) → {"PASS" if c1_pass else "FAIL"}')
print(f'  c2 loss_flow converged:  FQL loss_flow={fql_loss_flow:.4f} '
      f'(threshold < 0.05) → {"PASS" if c2_pass else "FAIL"}')
print(f'  c3 actor_loss OOM:       |FQL|/|ReBRAC|={ratio:.3f} '
      f'(threshold ∈ [0.1, 10]) → {"PASS" if c3_pass else "FAIL"}')
print(f'  c4 monotonicity:         FQL last-30% slope_mean={fql_slope:+.5f} '
      f'(threshold ≥ 0) → {"PASS" if c4_pass else "FAIL"}')
print()
print(f'  OVERALL: {"Gate B PASS" if verdict_block["overall_pass"] else "Gate B FAIL"}')

# ---- Write verdict + overview to results/ ----
verdict_path = SUMMARIES_DIR / 'gate_b_verdict.json'
verdict_path.write_text(json.dumps(summary, indent=2, default=str))
print(f'\n[wrote] {verdict_path}')

csv_path = SUMMARIES_DIR / 'gate_b_overview.csv'
with csv_path.open('w', newline='') as fp:
    w = csv.writer(fp)
    w.writerow(['algo', 'seed', 'last3_eval_success_mean', 'last30pct_eval_slope',
                 'loss_flow_last_5pct_mean', 'actor_loss_last_5pct_mean', 'n_eval_points_dedup'])
    for algo in ['rebrac', 'fql']:
        for seed in sorted(per_algo[algo].keys()):
            s = per_algo[algo][seed]
            w.writerow([algo, seed,
                         f'{s["last3_eval_success_mean"]:.4f}',
                         f'{s["last30pct_eval_slope"]:+.5f}',
                         f'{s["loss_flow_last_5pct_mean"]:.4f}',
                         f'{s["actor_loss_last_5pct_mean"]:+.4f}',
                         s['n_eval_points_dedup']])
    # Append aggregated row per algo
    for algo in ['rebrac', 'fql']:
        a = agg_by_algo[algo]
        w.writerow([algo, f'mean({a["seeds"]})',
                     f'{a["last3_mean"]:.4f}',
                     f'{a["last30pct_slope_mean"]:+.5f}',
                     f'{a["loss_flow_mean"]:.4f}',
                     f'{a["actor_loss_mean"]:+.4f}',
                     '—'])
print(f'[wrote] {csv_path}')


## 5.5 Final 100-ep test_result.json side-by-side (供 paper anchor 引用)

Gate B 判据 c1 用的是 in-training last-3 eval mean（spec §5.3 强调避免 single-shot fluctuation）。但 paper 写作时通常 cite **single canonical final-eval number**——把 §3 写入的 `test_result.json` 也列出来便于参考。

In [ ]:
import json
print('=' * 100)
print(f'{"algo":<8} {"seed":>5} {"success":>10} {"return":>10} {"safety":>10} {"prog":>10} {"path_eff":>10}')
print('-' * 100)
for r in RUNS:
    test_json = Path(r['result_dir']) / 'test_result.json'
    if not test_json.exists():
        print(f'{r["algo"]:<8} {r["seed"]:>5} MISSING')
        continue
    d = json.loads(test_json.read_text())
    # Bug 3 fix: actual field names are eval_* not eval_avg_*
    succ = d.get('eval_success_rate', float('nan'))
    ret_ = d.get('eval_return', float('nan'))
    safety = d.get('eval_safety_cost', float('nan'))
    prog = d.get('eval_progress_ratio', float('nan'))
    path_eff = d.get('eval_path_efficiency', float('nan'))
    print(f'{r["algo"]:<8} {r["seed"]:>5} {succ:>10.4f} {ret_:>10.2f} {safety:>10.3f} {prog:>10.4f} {path_eff:>10.4f}')
print('=' * 100)
print('\nReBRAC anchor reference (broad val v2 N0, crosscomp dataset, 2 seed): 0.850 ± 0.024')
print('FQL Gate B target on this dataset: ≥ ReBRAC last-3 − 0.08 (aggregated across seeds)')
print('Caveat (Bug 2): num_eval_episodes is bounded by manifest (30 ep), not --episodes 100')


## 6. 跑完后清单（回到 local）

### Sync 回 git 仓库（只取 `results/`）

```bash
# 在本地 repo root：
rsync -av '<drive>/results/offline/fql_succession/gate_b/' \
  results/offline/fql_succession/gate_b/
```

包含：
- `results/offline/fql_succession/gate_b/{rebrac,fql}_e_uni_seed{42,0}/test_result.json` × 4
- `results/offline/fql_succession/gate_b/{rebrac,fql}_e_uni_seed{42,0}/eval_log.csv` × 4
- `results/offline/fql_succession/gate_b/summaries/{gate_b_verdict.json, gate_b_overview.csv}`

**checkpoint 不动**（agent_final.pt 大文件，gitignore；只在 P2 spec 想 reuse seed=42/0 anchor 时回到 Drive）。

### 后续步骤

1. **写** `docs/fql_succession_gate_b_report.md`（升级 interim report → final report）：
   - 4 runs × 2 seed aggregated 数字
   - 4 criteria verdict + 每 seed 单独 breakdown
   - 与 broad val v2 N0 anchor 对比（dataset / protocol 差异分析）
   - 与 rev.1 single-seed interim report 数字对照（验证 Bug 1 fix 实际影响）
2. **Verdict-conditional 分支**：
   - **Gate B aggregated PASS**：升级 `docs/fql_succession_plan_v0.md` 到 v1.1，开始 P2 main comparison spec
   - **c4 FAIL（双 seed 都呈负 slope）**：systematic 末段不稳定 → Option D (FQL stability ablation：critic LN / tau=0.001 / 100k step / actor warm-up)
   - **c1 FAIL（FQL << ReBRAC across seeds）**：implementation issue → debug 3 iter；仍失败 STOP (plan v1 R4 mitigation)
   - **混合 (c4 一 seed fail / 一 seed pass)**：单点 noise，标 caveat 进 P2，做 3-seed extension 看是否复现
3. **Bug 2 处理**：决定生成 `single_u10_cross_tgt15_ep100` 大 manifest，或修 `evaluate_offline` 让 `--episodes` 不被 manifest 覆盖；P2 spec 起草时定
4. **Spec CLI rename** (deferred)：Task E 闭环后 patch `docs/fql_succession_p0p1_spec.md` §5.1/§5.2 字段名对齐实际代码
5. **N0 1000-ep audit**（可选 paper appendix evidence）：Gate B 顺利通过后，用 §Task A audit 工具在 N0 cell 跑 1000-ep audit 作为 paper §appendix evidence